# E01 Matrix Sensing Benchmark - PyTorch

Notebook-first Matrix Sensing experiment. The autograd problem lives in `problems/`; this notebook owns the experiment grid, optimizer step, run loop, joblib parallelism, plots, and conclusion.

## Imports

In [ ]:
from pathlib import Path
import importlib
import os
import sys
import time
from functools import partial

for name in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ.setdefault(name, "1")

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from joblib import Parallel, delayed
from tqdm.auto import tqdm

PROJECT = Path.cwd().resolve()
if not (PROJECT / "problems").exists():
    PROJECT = PROJECT.parent.resolve()
sys.path.insert(0, str(PROJECT))

from optimizers import MuonExact, Shampoo


def fresh_import(module_name):
    importlib.invalidate_caches()
    sys.modules.pop(module_name, None)
    package_name, attr_name = module_name.rsplit(".", 1)
    package = importlib.import_module(package_name)
    if hasattr(package, attr_name):
        delattr(package, attr_name)
    return importlib.import_module(module_name)

matrix_construction = fresh_import("problems.MatrixConstruction")
matrix_sensing = fresh_import("problems.MatrixSensing")
randn = matrix_construction.randn
make_matrix_sensing_problem = matrix_sensing.make_matrix_sensing_problem

torch.set_default_dtype(torch.float64)
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print(f"project = {PROJECT}")
print(f"torch   = {torch.__version__}")
print(f"joblib  = {joblib.__version__}")

## Parameters And Run Grid

`RUN_SPECS` is the exact list submitted to joblib. `GRID_PREVIEW` shows one row per run.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE_NAME = "float64"
ALGOS = ["Muon", "Muon-Exact", "Shampoo", "Adam", "SGD"]
DIMS = [50, 60, 70]
SEEDS = list(range(10))

BASE_SPEC = dict(
    rank=5,
    lr=0.01,
    noise=0.0,
    dist="normal",
    spectrum="hard-cutoff",
    kappa=1.0,
    init_scale=0.01,
    iters=2000,
    epsilon=0.01,
    device_type=DEVICE.type,
    dtype_name=DTYPE_NAME,
)
NUM_WORKERS = min(8, os.cpu_count() or 1)
JOBLIB_BACKEND = "loky"

RUN_SPECS = [
    {**BASE_SPEC, "algo": algo, "d": d, "seed": seed}
    for algo in ALGOS
    for d in DIMS
    for seed in SEEDS
]

GRID_PREVIEW = pd.DataFrame(RUN_SPECS)
GRID_PREVIEW.insert(0, "run_id", range(len(GRID_PREVIEW)))
GRID_PREVIEW.insert(4, "m_meas", 2 * GRID_PREVIEW["d"] * GRID_PREVIEW["rank"])
GRID_PREVIEW = GRID_PREVIEW[[
    "run_id", "algo", "d", "rank", "m_meas", "seed", "lr", "iters", "epsilon",
    "noise", "dist", "spectrum", "kappa", "init_scale", "device_type", "dtype_name",
]]

print(f"device={DEVICE}, workers={NUM_WORKERS}, backend={JOBLIB_BACKEND}")
print(f"runs={len(RUN_SPECS)}, iters={BASE_SPEC['iters']}, total_steps={len(RUN_SPECS) * BASE_SPEC['iters']}")
display(GRID_PREVIEW)

## Problem

Target matrix:

$$
X^\star = U \operatorname{diag}(s)V^	op
$$

Measurements:

$$
y_i = \langle A_i, X^\starangle + arepsilon_i
$$

Loss:

$$
f(X) = rac{1}{2m}\sum_{i=1}^{m}(\langle A_i, Xangle-y_i)^2
$$

## Step

In [ ]:
def step(state):
    problem, x, opt = state["problem"], state["x"], state["optimizer"]
    opt.zero_grad(set_to_none=True)
    loss = problem.loss(x)
    loss.backward()
    grad_norm = float(x.grad.detach().norm().cpu())
    opt.step()
    return float(loss.detach().cpu()), grad_norm


STEP_FN = step

## Runtime Helpers

In [ ]:
def dtype_from_name(name):
    dtype = getattr(torch, name)
    if not isinstance(dtype, torch.dtype):
        raise ValueError(f"unknown torch dtype: {name}")
    return dtype


def configure_torch(dtype):
    torch.set_default_dtype(dtype)
    torch.set_num_threads(1)
    try:
        torch.set_num_interop_threads(1)
    except RuntimeError:
        pass


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def make_optimizer(algo, params, lr, rank):
    if algo == "Muon" and hasattr(torch.optim, "Muon"):
        return torch.optim.Muon(params, lr=lr, weight_decay=0.0, momentum=0.9, nesterov=False, ns_steps=5)
    if algo == "Muon":
        return MuonExact(params, lr=lr, momentum=0.9, variant="newton_schulz", ns_steps=5)
    if algo in {"Muon-Exact", "MuonExact"}:
        return MuonExact(params, lr=lr, momentum=0.9, variant="exact")
    if algo == "Shampoo":
        return Shampoo(params, lr=lr, beta2=0.9, epsilon=1e-8)
    if algo == "Adam":
        return torch.optim.Adam(params, lr=lr)
    if algo == "SGD":
        return torch.optim.SGD(params, lr=lr, momentum=0.9)
    raise ValueError(f"unknown algo: {algo}")

## Single Run

In [ ]:
def run_one(spec, step_fn=STEP_FN):
    device = torch.device(spec["device_type"])
    dtype = dtype_from_name(spec["dtype_name"])
    configure_torch(dtype)

    problem = make_matrix_sensing_problem(
        spec["d"], spec["rank"], noise=spec["noise"], dist=spec["dist"],
        spectrum=spec["spectrum"], kappa=spec["kappa"], seed=spec["seed"],
        device=device, dtype=dtype,
    )
    x = torch.nn.Parameter(
        randn((spec["d"], spec["d"]), spec["seed"] + 3000, device, dtype) * spec["init_scale"]
    )
    opt = make_optimizer(spec["algo"], [x], spec["lr"], spec["rank"])
    state = {"problem": problem, "x": x, "optimizer": opt}

    losses, grad_norms, k_eps = [], [], None
    sync(device)
    t0 = time.time()
    for i in range(spec["iters"]):
        loss, grad_norm = step_fn(state)
        losses.append(loss)
        grad_norms.append(grad_norm)
        if k_eps is None and loss <= spec["epsilon"]:
            k_eps = i + 1
    sync(device)

    row = {
        **{k: spec[k] for k in ["algo", "d", "lr", "noise", "dist", "spectrum", "kappa", "init_scale", "seed", "iters"]},
        "problem": "MatrixSensing",
        "r": spec["rank"],
        "m_meas": problem.m_meas,
        "final_loss": losses[-1],
        "min_loss": min(losses),
        "K_epsilon": k_eps or spec["iters"] + 1,
        "time_s": time.time() - t0,
    }
    return (spec["algo"], spec["d"], spec["seed"]), row, {"loss": losses, "grad_norm": grad_norms}


RUN_ONE = partial(run_one, step_fn=STEP_FN)

## Run Experiment

In [ ]:
def sort_results(df):
    if df.empty:
        return df
    df = df.copy()
    df["algo"] = pd.Categorical(df["algo"], categories=ALGOS, ordered=True)
    df = df.sort_values(["algo", "d", "seed"]).reset_index(drop=True)
    df["algo"] = df["algo"].astype(str)
    return df


def run_experiment(specs=RUN_SPECS, run_one=RUN_ONE, num_workers=NUM_WORKERS, backend=JOBLIB_BACKEND):
    specs = [spec.copy() for spec in specs]
    num_workers = max(1, min(int(num_workers), len(specs)))
    if num_workers == 1:
        results = (run_one(spec) for spec in specs)
    else:
        results = Parallel(n_jobs=num_workers, backend=backend, return_as="generator_unordered")(
            delayed(run_one)(spec) for spec in specs
        )

    rows, trajectories = [], {}
    desc = "E01 runs" if num_workers == 1 else f"E01 runs ({num_workers} joblib)"
    for key, row, traj in tqdm(results, total=len(specs), desc=desc, unit="run"):
        rows.append(row)
        trajectories[key] = traj
    return sort_results(pd.DataFrame(rows)), trajectories

## Plotting API

In [ ]:
from plotting import (
    ordered_algos_in,
    ordered_dims_in,
    plot_algorithm_dimension_grid,
    plot_algorithms_for_dimension,
    plot_all_mean_curves_combined,
    plot_color_key,
    plot_dimensions_for_algorithm,
    plot_metric_bar,
    plot_metric_overview,
    plot_seed_variability_for_dimension,
    summary_table,
)


def show_figure(fig):
    display(fig)
    plt.close(fig)

## Execute

In [ ]:
df, trajectories = run_experiment()

## Result Tables

In [ ]:
summary = summary_table(df)
display(df.sort_values(["algo", "d", "seed"]).reset_index(drop=True))
display(summary)

## Color Key

In [ ]:
fig, axes = plot_color_key(df)
show_figure(fig)

## Metric Overview

In [ ]:
fig, axes = plot_metric_overview(df)
show_figure(fig)

## K Epsilon Bar

In [ ]:
fig, ax = plot_metric_bar(df, "K_epsilon_mean", "Mean K_epsilon by algorithm and dimension", "iterations")
show_figure(fig)

## Wall Clock Bar

In [ ]:
fig, ax = plot_metric_bar(df, "time_s_mean", "Mean wall-clock by algorithm and dimension", "seconds")
show_figure(fig)

## Minimum Loss Bar

In [ ]:
fig, ax = plot_metric_bar(df, "min_loss_mean", "Mean minimum loss by algorithm and dimension", "loss", log_y=True)
show_figure(fig)

## Final Loss Bar

In [ ]:
fig, ax = plot_metric_bar(df, "final_loss_mean", "Mean final loss by algorithm and dimension", "loss", log_y=True)
show_figure(fig)

## Same Dimension Algorithm Comparisons

In [ ]:
for d in ordered_dims_in(df):
    fig, ax = plot_algorithms_for_dimension(trajectories, d)
    show_figure(fig)

## Same Algorithm Dimension Comparisons

In [ ]:
for algo in ordered_algos_in(df):
    fig, ax = plot_dimensions_for_algorithm(trajectories, algo)
    show_figure(fig)

## All Mean Curves

In [ ]:
fig, ax = plot_all_mean_curves_combined(trajectories)
show_figure(fig)

## Algorithm-Dimension Grid

In [ ]:
fig, axes = plot_algorithm_dimension_grid(trajectories)
show_figure(fig)

## Seed Variability By Dimension

In [ ]:
for d in ordered_dims_in(df):
    fig, ax = plot_seed_variability_for_dimension(trajectories, d)
    show_figure(fig)

## Conclusion

In [ ]:
def conclusion_markdown(df):
    summary = summary_table(df)
    lines = [
        "### Result Summary",
        "",
        f"- Runs: `{len(df)}`",
        f"- Methods: `{', '.join(sorted(df['algo'].unique()))}`",
        f"- Iterations per run: `{int(df['iters'].iloc[0])}`",
        f"- Total optimizer steps: `{len(df) * int(df['iters'].iloc[0])}`",
        "",
    ]
    for d in sorted(df["d"].unique()):
        sub = summary[summary["d"] == d]
        best_k = sub.loc[sub["K_epsilon_mean"].idxmin()]
        best_time = sub.loc[sub["time_s_mean"].idxmin()]
        best_loss = sub.loc[sub["min_loss_mean"].idxmin()]
        lines.append(
            f"- d={d}: best K_epsilon is `{best_k.algo}` at `{best_k.K_epsilon_mean:.1f}` steps; "
            f"fastest wall-clock is `{best_time.algo}` at `{best_time.time_s_mean:.3f}s`; "
            f"lowest min_loss is `{best_loss.algo}` at `{best_loss.min_loss_mean:.3e}`."
        )
    lines.append("")
    lines.append("Muon vs Muon-Exact checks whether the Newton-Schulz approximation tracks the exact SVD polar direction.")
    lines.append("Shampoo is included as a second-order preconditioned baseline; compare both K_epsilon and wall-clock.")
    return "\n".join(lines)


display(Markdown(conclusion_markdown(df)))